# MMELON Model Fine-Tuning

Fine-tune the MMELON (Multi-Modal Encoder for Learning on Molecules) model on hit prediction data.

## Overview

This notebook demonstrates how to fine-tune MMELON for binary classification tasks in drug discovery:

**What is MMELON?**
- Multi-modal molecular encoder combining multiple molecular representations
- Integrates graph neural networks, fingerprints, and other molecular features
- Pre-trained on large-scale molecular datasets

**Workflow:**
1. **Setup & Configuration** - Environment setup and hyperparameter configuration
2. **Data Preparation** - Load screening data and create data loaders
3. **Model Initialization** - Load pre-trained MMELON and configure prediction head
4. **Training Setup** - Configure PyTorch Lightning trainer
5. **Fine-Tuning** - Train the model on your dataset
6. **Evaluation** - Assess model performance and save results

**Use Cases:**
- Hit prediction from screening campaigns
- Activity classification (active/inactive)
- Lead optimization prioritization

In [ ]:
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False
IN_COLAB = in_colab()
print(f"{IN_COLAB=}")

IN_COLAB=False


## Installation Instructions

### Local Environment
```bash
conda create -n mmelon311 python=3.11 -y
conda activate mmelon311
git clone https://github.com/jmorrone/biomed-multi-view.git
pip install git+https://github.com/jmorrone/biomed-multi-view.git
pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.1.0 torchvision==0.16.0
pip install -f https://data.pyg.org/whl/torch-2.1.0+cu121.html "pyg_lib==0.4.0+pt21cu121" "torch_scatter==2.1.2+pt21cu121" "torch_cluster==1.6.3+pt21cu121" "torch_spline_conv==1.2.2+pt21cu121" --upgrade-strategy only-if-needed
pip install -q notebook ipykernel scikit-learn pandas openpyxl rdkit
```

In [ ]:
if IN_COLAB:
    print("***** Select runtime 2025.07!!!")
    !git clone https://github.com/jmorrone/biomed-multi-view.git

In [ ]:
if IN_COLAB:
    !pip install git+https://github.com/jmorrone/biomed-multi-view.git  

In [ ]:
if IN_COLAB:
    import os
    os.chdir('biomed-multi-view')
    !pip install -r requirements.txt

## 1. Setup and configuration

Edit these paths and hyperparameters according to your needs.

In [ ]:
import os
run_on_wdr91 = True
USER = os.getenv("USER")  
  
if IN_COLAB:
    MODELS_DIR = "/content/models/mmelon"
    DEMO_SIZE = 100
else:
    USER = os.getenv("USER")
    MODELS_DIR = f"/proj/ligand_ai/{USER}/models/mmelon"
    DEMO_SIZE = 100 # None

if run_on_wdr91:
    NAME = "wdr91_asms"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        # DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1"
        DATA_PATH = f"/proj/ligand_ai/users/{USER}/data/wdr91_ASMS_train_val_v1"
else:
    NAME = "pgk2_del_cdd"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        # DATA_PATH = f"/proj/ligand_ai/users/{USER}/data/PGK2_DEL_cdd_to_creative"
        DATA_PATH = f"/proj/ligand_ai/users/{USER}/data/PGK2_DEL_cdd_to_creative"

MODEL_DIR = os.path.join(MODELS_DIR, NAME)
if DEMO_SIZE is not None:
    MODEL_DIR += f"_demo_{DEMO_SIZE}"
    DEMO_DATA_PATH = DATA_PATH + f"_demo_{DEMO_SIZE}" 
    os.makedirs(DEMO_DATA_PATH, exist_ok=True)    

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
EPOCHS = 1 # 10  # Number of training epochs
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
SEED = 42  # Random seed for reproducibility

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = None  # None for auto-detection, or specify 'cuda' or 'cpu'

print("Configuration loaded successfully!")
print(f"Output model directory: {MODEL_DIR}")
print(f"Base Model: {BASE_MODEL}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

Configuration loaded successfully!
Output model directory: /proj/ligand_ai/ozery/models/mmelon/wdr91_asms_demo_100
Base Model: ibm-research/biomed.sm.mv-te-84m
Training epochs: 1
Batch size: 16
Learning rate: 2e-05


## 2. Load and Prepare Data

Load the dataset splits and prepare them for training.

In [ ]:
import numpy as np
import pandas as pd
def reduce_data_for_demo(df: pd.DataFrame, demo_size: int, label_column: str, seed: int=42)-> pd.DataFrame:
          
    # Sample at most demo_size samples per class (label=0 and label=1)
    df_list = []
    for label in df[label_column].unique():
        df_label = df[df[label_column] == label]
        n = len(df_label)
        if n > demo_size:
            df_label = df_label.sample(n=demo_size, random_state=seed)
            print(f"Class {label}: {n} samples reduced to {len(df_label)} samples for demo.")
        else:
            print(f"Class {label}: {n} samples (no reduction needed for demo).")
        df_list.append(df_label)

    # Combine and shuffle
    df = pd.concat(df_list).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return df

In [ ]:
import pandas as pd
import numpy as np
# Load split data

split_path = DATA_PATH

print(f"Loading split from: {split_path}")

data_dir = DATA_PATH if DEMO_SIZE is None else DEMO_DATA_PATH
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"{split_path}/{split}.csv")
    
    if DEMO_SIZE is not None:
        original_size = len(df)
        df = reduce_data_for_demo(df, demo_size=DEMO_SIZE, label_column='label', seed=42)
        print(f"***** DEMO MODE: Reduced {split} set from {original_size} to {len(df)} samples (max {DEMO_SIZE} per class)")
    
    output_file = f"{data_dir}/data_{split}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved {output_file} for MMELON fine-tuning.")
    print(f"***  {split}: {len(df):,} samples ({df['label'].mean()*100:.2f}% positive)")

Loading split from: /proj/ligand_ai/users/ozery/data/wdr91_ASMS_train_val_v1
Class 0: 305208 samples reduced to 100 samples for demo.
Class 1: 124 samples reduced to 100 samples for demo.
***** DEMO MODE: Reduced train set from 305332 to 200 samples (max 100 per class)
Saved /proj/ligand_ai/users/ozery/data/wdr91_ASMS_train_val_v1_demo_100/data_train.csv for MMELON fine-tuning.
***  train: 200 samples (50.00% positive)
Class 0: 33912 samples reduced to 100 samples for demo.
Class 1: 14 samples (no reduction needed for demo).
***** DEMO MODE: Reduced val set from 33926 to 114 samples (max 100 per class)
Saved /proj/ligand_ai/users/ozery/data/wdr91_ASMS_train_val_v1_demo_100/data_val.csv for MMELON fine-tuning.
***  val: 114 samples (12.28% positive)
Class 1: 58 samples (no reduction needed for demo).
Class 0: 176 samples reduced to 100 samples for demo.
***** DEMO MODE: Reduced test set from 234 to 158 samples (max 100 per class)
Saved /proj/ligand_ai/users/ozery/data/wdr91_ASMS_train_v

## 3. Prepare Data Loaders

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel

/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-09 11:01:47,881 - rdkit - INFO - teddy:22986827281088:0:0 - Enabling RDKit 2022.09.5 jupyter extensions


In [ ]:
# Set random seed for reproducibility
pl.seed_everything(SEED)

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*80)
print("INITIALIZING MMELON MODEL")
print("="*80)

# Dataset arguments for MMELON framework
dataset_args = {
    'task_type': TaskType.CLASSIFICATION,
    'num_tasks': 1,  # Single task for binary classification
    'modalities': ['TEXT_MODEL', 'IMAGE_MODEL', 'GRAPH_2D_MODEL'],
    'smiles_col': 'smiles',
    'label_cols': ['label'],
    'split_col': None
}

# Create data modules
print(f"Creating data modules using data in {data_dir}...")
train_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='train'
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
    num_workers=4
)

val_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='val'
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=val_dataset.collate_fn,
    num_workers=4
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Seed set to 42
2026-03-09 11:01:49,985 - root - INFO - teddy:22986827281088:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-09 11:01:49,986 - root - INFO - teddy:22986827281088:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


INITIALIZING MMELON MODEL
Creating data modules using data in /proj/ligand_ai/users/ozery/data/wdr91_ASMS_train_val_v1_demo_100...


2026-03-09 11:01:53,648 - root - INFO - teddy:22986827281088:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-09 11:01:53,650 - root - INFO - teddy:22986827281088:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


Train batches: 13
Val batches: 8


## 4. Initialize MMELON Model

In [ ]:
FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

# Model parameters
model_params = {
    'agg_arch': FUSION_STRATEGY.value[0],
    'agg_gate_input': FUSION_STRATEGY.value[1],
    'agg_weight_freeze': FUSION_STRATEGY.value[2],
    'inference_mode': False
}

# Finetuning arguments
finetuning_args = {
    'weight_freeze': 'unfrozen',
    'initialization': 'default',
    'head_arch': 'mlp',
    'use_norm': True,
    'head_dropout': 0.2
}

# Create Lightning module
print("Creating Lightning module...")
lightning_module = FineTuneLightningModule(
    base_model_class='bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView',
    model_params=model_params,
    task_type='classification',
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args=finetuning_args
)

# Load pretrained weights from HuggingFace
print(f"Loading pretrained model from HuggingFace: {BASE_MODEL}")
pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY,
    model_path=BASE_MODEL,
    huggingface=True
)

lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)

print("Model initialized successfully!")

2026-03-09 11:01:54,288 - root - INFO - teddy:22986827281088:0:0 - Initializing model with params {'agg_arch': 'coeff_mlp', 'agg_gate_input': 'coeff_mlp', 'agg_weight_freeze': None, 'inference_mode': False}


Creating Lightning module...


/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2026-03-09 11:01:55,322 - root - INFO - teddy:22986827281088:0:0 - Using coeff_mlp architecture for aggregator
2026-03-09 11:01:55,323 - root - INFO - teddy:22986827281088:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-09 11:01:55,340 - root - INFO - teddy:22986827281088:0:0 - Initialized the model with random weights as checkpoint_path was specified as None
2026-03-09 11:01:55,341 - root - INFO - teddy:2

Loading pretrained model from HuggingFace: ibm-research/biomed.sm.mv-te-84m


2026-03-09 11:01:56,311 - root - INFO - teddy:22986827281088:0:0 - Using coeff_mlp architecture for aggregator
2026-03-09 11:01:56,311 - root - INFO - teddy:22986827281088:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-09 11:02:00,231 - root - INFO - teddy:22986827281088:0:0 - in train False setting deterministic_eval = True


Model initialized successfully!


## 5. Configure Trainer

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

# Setup callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=MODEL_DIR,
        filename='mmelon-{epoch:02d}-{val_loss:.2f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
        every_n_epochs=1
    )
]

# Create trainer
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=callbacks,
    default_root_dir=MODEL_DIR,
    accelerator='auto',
    devices=1,
    log_every_n_steps=10,
    val_check_interval=1.0,
    enable_checkpointing=True,
    enable_model_summary=True
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


## 6. Fine-Tune Model

In [ ]:
%%time

print("="*80)
print("STARTING TRAINING")
print("="*80)

trainer.fit(lightning_module, train_loader, val_loader)

print("="*80)
print("TRAINING COMPLETE!")
print("="*80)

STARTING TRAINING


You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /gpfs/ZuRust/proj/ligand_ai/ozery/models/mmelon/wdr91_asms_demo_100 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                    | Params | Mode  | FLOPs
----------------------------------------------------------------------
0 | model     | SmallMoleculeMultiView  | 84.1 M | train | 0    
1 | pred_head | MultiTaskPredictionHead | 461 K  | train | 0    
2 | criterion | BCEWithLogitsLoss       | 0      | train | 0    
-------------------

Sanity Checking: |                                                                   | 0/? [00:00<?, ?it/s]

2026-03-09 11:02:05,589 - root - INFO - teddy:22986827281088:0:0 - in train False setting deterministic_eval = True


Sanity Checking DataLoader 0:  50%|█████████████████████                     | 1/2 [00:04<00:04,  0.24it/s]

...

Epoch 0: 100%|██████████████████████████████████| 13/13 [00:03<00:00,  4.07it/s, v_num=1, train_loss=0.621]

2026-03-09 11:02:14,122 - root - INFO - teddy:22986827281088:0:0 - in train False setting deterministic_eval = True


Epoch 0: 100%|██████████████████| 13/13 [00:04<00:00,  3.13it/s, v_num=1, train_loss=0.621, val_loss=0.503]

/proj/bmfm/users/ozery/miniforge3/envs/mmelon311/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████████████| 13/13 [00:08<00:00,  1.50it/s, v_num=1, train_loss=0.621, val_loss=0.503]
TRAINING COMPLETE!
CPU times: user 4.7 s, sys: 5.04 s, total: 9.74 s
Wall time: 18 s


In [ ]:
os.listdir(MODEL_DIR)

['lightning_logs', 'mmelon-epoch=00-val_loss=0.50.ckpt', 'last.ckpt']